# La ligne de commande

<svg xmlns="http://www.w3.org/2000/svg" width="480" height="200" viewBox="0 0 600 250">
<rect x="60" y="20" width="480" height="170" rx="14" fill="#2d2f33"/>
<rect x="60" y="20" width="480" height="34" rx="14" fill="#44474d"/>
<rect x="60" y="40" width="480" height="14" fill="#44474d"/>
<circle cx="86" cy="37" r="7" fill="#ff5f57"/><circle cx="110" cy="37" r="7" fill="#febc2e"/><circle cx="134" cy="37" r="7" fill="#28c840"/>
<text x="92" y="105" fill="#28c840" font-family="monospace" font-size="34" font-weight="bold">$</text>
<text x="128" y="105" fill="#f2f2f2" font-family="monospace" font-size="34">ls -lh</text>
<text x="92" y="152" fill="#9aa0a6" font-family="monospace" font-size="34">patients.csv</text>
<rect x="92" y="170" width="20" height="5" fill="#28c840"/>
<text x="300" y="235" fill="#2d2f33" font-family="Helvetica, Arial, sans-serif" font-size="26" font-weight="bold" text-anchor="middle">The shell</text>
</svg>

[Chaque question est suivie d'une cellule qui commence déjà par `%%bash` : écrivez là votre commande, sous les lignes qui sont données. Tout dans une telle cellule est géré par bash, pas par Python.]

Pour comprendre ce que fait une commande, collez-la dans [explainshell.com](https://explainshell.com/) : elle explique chaque partie de la commande.

Deux choses à savoir avant de commencer :

- Chaque cellule est un **nouveau shell**, donc un `cd` dans une cellule est oublié dans la suivante. C'est pourquoi les cellules travaillant sur les données recommencent avec `cd ../data`.
- Les données sont le dossier `../data`, un niveau supérieur : un export synthétique depuis un service hospitalier (`patients.csv`, un tracé ECG par fichier en `ecg/`, des notes en texte libre en `notes/`, et le journal de l'export en `export.log`).

Exécutez d’abord la cellule ci-dessous. C'est la seule cellule **Python** du notebook : elle construit `../data` si le dossier n'est pas encore là. Tout est synthétique et ensemencé, c'est donc la même exportation pour tout le monde.

In [1]:
from pathlib import Path
from subprocess import run
from sys import executable

data = Path("../data")
if (data / "patients.csv").exists():
    print(f"data already there: {data.resolve()}")
else:
    run([executable, "../../scripts/make_data_c01.py", str(data)], check=True)
    print(f"data generated in {data.resolve()}")

data already there: /home/paul/Documents/Cours/MastereSpecialiseSanteNumerique2026/Programming01/data


## 1. Où suis-je et qu'y a-t-il ici ?

Imprimez le dossier dans lequel vous vous trouvez (`pwd`), puis listez ce qui se trouve à côté de ce cahier (`ls`).

In [2]:
%%bash
pwd
ls

/home/paul/Documents/Cours/MastereSpecialiseSanteNumerique2026/Programming01/Session03
clean.py
command_line.ipynb
command_line-solutions.ipynb
img
rows.py
slides.md
slides.pptx


Listez maintenant `../data`, un fichier par ligne, avec des tailles qu'un humain peut lire et avec les fichiers cachés (`-l`, `-h`, `-a`).

In [3]:
%%bash
cd ../data
ls -lha

total 44K
drwxrwxr-x  5 paul paul 4.0K Oct 10 19:30 .
drwxrwxr-x 10 paul paul 4.0K Oct 13 14:33 ..
drwxrwxr-x  2 paul paul 4.0K Sep  8 12:30 ecg
-rw-rw-r--  1 paul paul 2.0K Sep  8 12:30 export.log
drwxrwxr-x  2 paul paul 4.0K Sep  8 12:30 exports
-rw-rw-r--  1 paul paul    1 Oct 10 19:30 .gitignore
drwxrwxr-x  2 paul paul 4.0K Sep  8 12:30 notes
-rw-rw-r--  1 paul paul  11K Sep  8 12:30 patients.csv
-rw-rw-r--  1 paul paul  129 Sep  8 12:30 README.md


Combien de fichiers y a-t-il dans `ecg/` ?

*Astuce : `ls` imprime un nom par ligne lorsque sa sortie n'est pas un terminal, et `wc -l` compte les lignes.*

In [4]:
%%bash
cd ../data
ls ecg/ | wc -l

60


## 2. Regarder à l'intérieur d'un fichier

Comptez les lignes de `patients.csv` (`wc -l`), puis imprimez ses trois premières lignes et sa dernière ligne (`head`, `tail`).

Il y a 240 patients. Pourquoi `wc -l` n'est-il pas d'accord ?

In [5]:
%%bash
cd ../data
wc -l patients.csv
head -3 patients.csv
tail -1 patients.csv  # 241 lines: the header is a line too

241 patients.csv
patient_id,age,sex,ward,admission,los_days,outcome
P0001,66,F,ICU,2024-04-21,6.4,transferred
P0002,14,F,ICU,2024-04-14,10.6,transferred
P0240,,F,CARDIO,2024-12-10,1.5,discharged


`cat` imprime un fichier entier. Imprimez la note `notes/P0001.txt` avec `cat`.

Ensuite, exécutez `cat ecg/P0002.csv | wc -l` : combien de lignes `cat` auraient été imprimées dans le cahier ?

In [6]:
%%bash
cd ../data
cat notes/P0001.txt
cat ecg/P0002.csv | wc -l  # cat is fine for 10 lines, not for a signal

Ward round note for P0001.
Escalated to ICU overnight, sepsis suspected.
121


## 3. Garder les lignes que vous souhaitez : `grep`

Combien de lignes de `patients.csv` mentionnent `ICU` (`-c`) ? Imprimez-en trois en guise de chèque.

In [7]:
%%bash
cd ../data
grep -c 'ICU' patients.csv
grep 'ICU' patients.csv | head -3

76
P0001,66,F,ICU,2024-04-21,6.4,transferred
P0002,14,F,ICU,2024-04-14,10.6,transferred
P0003,73,M,ICU,2024-07-09,8.6,transferred


Combien de séjours **CARDIO** se sont terminés par le transfert du patient ? Enchaînez deux `grep` avec un tuyau : le premier se rétrécit, le second compte ce qui reste.

In [8]:
%%bash
cd ../data
grep 'CARDIO' patients.csv | grep -c 'transferred'

16


Quelles notes de service en `notes/` mentionnent le sepsis, quel que soit le cas ? Imprimez uniquement les noms de fichiers.

*Astuce : `-r` recherche un dossier entier, `-i` ignore la casse, `-l` imprime les noms au lieu de faire correspondre les lignes.*

In [9]:
%%bash
cd ../data
grep -rli 'sepsis' notes/

notes/P0012.txt
notes/P0001.txt
notes/P0006.txt
notes/P0005.txt


## 4. Colonnes, ordre, nombres : `cut`, `sort`, `uniq`

Imprimez la salle des cinq premiers patients.

*Astuce : `cut` nécessite le séparateur (`-d,`) et le numéro de champ (`-f4`) ; les champs sont numérotés à partir de 1.*

In [10]:
%%bash
cd ../data
cut -d, -f4 patients.csv | head -5

ward
ICU
ICU
ICU
PEDIA


Comptez les séjours par service, les plus fréquents en premier.

*Indice : `sort` place les lignes identiques les unes à côté des autres, `uniq -c` les réduit et les compte, et `sort -rn` ordonne les comptes. Supprimez d'abord l'en-tête avec `tail -n +2`, ce qui signifie « commencer à la ligne 2 ».*

In [11]:
%%bash
cd ../data
tail -n +2 patients.csv | cut -d, -f4 | sort | uniq -c | sort -rn

     76 ICU
     64 CARDIO
     39 ONCO
     35 PEDIA
     26 GERIA


Imprimez les cinq séjours les plus longs, le plus long en premier.

*Astuce : `sort` peut fonctionner sur une colonne indépendante : `-t,` définit le séparateur, `-k6` trie le champ 6, `-rn` l'ordonne sous forme de nombre, à l'envers.*

In [12]:
%%bash
cd ../data
tail -n +2 patients.csv | sort -t, -k6 -rn | head -5

P0221,59,F,ICU,2024-10-19,18.0,discharged
P0030,73,M,ICU,2024-02-14,17.4,discharged
P0174,62,F,ICU,2024-08-20,16.0,deceased
P0077,83,M,ICU,2024-11-06,16.0,deceased
P0192,63,M,ICU,2024-02-14,15.4,deceased


## 5. Trouver des fichiers

Combien de fichiers `.csv` y a-t-il sous `../data`, à n'importe quelle profondeur ? Imprimez-en cinq.

*Indice : `find . -name '*.csv'` parcourt l'arbre d'ici vers le bas. Les guillemets sont importants : sans eux, le shell se développe `*.csv` avant que `find` ne le voie.*

In [13]:
%%bash
cd ../data
find . -name '*.csv' | wc -l
find . -name '*.csv' | head -5

62
./ecg/P0048.csv
./ecg/P0030.csv
./ecg/P0060.csv
./ecg/P0003.csv
./ecg/P0007.csv


Un tracé ECG est arrivé vide de l'enregistreur. Trouvez-le (`-type f`, `-empty`).

In [14]:
%%bash
cd ../data
find . -type f -empty

./ecg/P0018.csv


Le dossier `exports/` contient un fichier dont le nom contient des espaces. Imprimez ses deux premières lignes.

*Indice : un espace sépare les arguments, le nom doit donc être cité, sinon le shell le lit comme trois arguments.*

In [15]:
%%bash
cd ../data
head -2 'exports/Patient Records 2024.csv'

patient_id,age,sex,ward,admission,los_days,outcome
P0001,66,F,ICU,2024-04-21,6.4,transferred


## 6. Écrire les résultats

Ecrivez le tableau des séjours par service en `../results/wards.txt`, puis imprimez le fichier pour le vérifier.

*Astuce : `>` envoie la sortie vers un fichier au lieu de l'écran, et `mkdir -p` crée le dossier (et ne se plaint pas s'il est déjà là).*

In [16]:
%%bash
cd ../data
mkdir -p ../results
tail -n +2 patients.csv | cut -d, -f4 | sort | uniq -c | sort -rn > ../results/wards.txt
cat ../results/wards.txt

     76 ICU
     64 CARDIO
     39 ONCO
     35 PEDIA
     26 GERIA


Écrivez les lignes ICU de `patients.csv` dans `../results/icu.csv`, avec l'en-tête sur la première ligne. Vérifiez ensuite que le fichier comporte le nombre de lignes souhaité.

*Indice : `>` écrase, `>>` ajoute.*

In [17]:
%%bash
cd ../data
head -1 patients.csv > ../results/icu.csv
grep 'ICU' patients.csv >> ../results/icu.csv
wc -l ../results/icu.csv

77 ../results/icu.csv


## 7. Appeler Python depuis le shell

Où se trouve le programme `python3` et de quelle version s'agit-il ? Ensuite, laissez bash le démarrer sur un programme d'une seule ligne.

*Indice : `which`, `python3 --version` et `python3 -c '...'`. Les guillemets simples remettent le programme à Python intact.*

In [18]:
%%bash
which python3
python3 --version
python3 -c 'print("hello from python, started by bash")'

/home/paul/Documents/Cours/MastereSpecialiseSanteNumerique2026/.venv/bin/python3
Python 3.14.8
hello from python, started by bash
